# Quantum caustics in the transverse-field Ising model

This notebook reproduces the caustic of Singh Roy et al. (2026) at a size that runs in a few
minutes, calling the package rather than reimplementing the physics. A single spin flipped at
the centre of the chain spreads outward as a light cone; near its front the magnetisation
concentrates into a squared-Airy interference pattern, the caustic. The steps are: evolve the
chain and form the caustic, cross-check it against the exact engine at small N, and show
the boundary effect that the subtraction removes.

Every quantity comes from a function in `src/`: `caustic_difference`, `compare_caustic`,
`evolve_caustic`, and `colormap`. Nothing here is reimplemented.

In [ ]:
using QuantumCaustics
import Plots

## Parameters

All parameters sit in the next cell. `Jxx < hz` places the chain in the paramagnetic phase,
where the fringe spacing carries the universal 2/3 scaling. `Jzz` is the integrability-breaking
coupling of Eq. 23; set it to a small nonzero value to watch the caustic survive the
perturbation.

In [ ]:
N   = 41      # spins, odd so the chain has a centre
Jxx = 0.5     # Ising coupling; Jxx < hz is the paramagnetic phase
hz  = 1.0     # transverse field
Jzz = 0.0     # integrability-breaking ZZ coupling; 0 is the pure TFIM
dt  = 0.1     # Trotter step
T   = 60.0    # total evolution time
(; N, Jxx, hz, Jzz, dt, T)   # the parameters, shown back as a double-check

## The caustic

`caustic_difference` evolves the chain twice, once with the central spin flipped and once
without, and returns the difference `dZ` of Eq. 22, in which any boundary contribution cancels.
The colour map of `dZ` against site and time is the light cone, a small version of Fig. 2. The
first two fringes at each site set the order parameter whose scaling is the paper's result.

In [ ]:
spec = TransverseFieldIsing(; Jxx = Jxx, hz = hz, Jzz = Jzz, boundary = OPEN)
res  = caustic_difference(spec; N = N, dt = dt, ttotal = T)
colormap(res.run; data = res.dZ, clabel = "dZ")

## Cross-check against the exact engine

The same chain at `N = 9` is evolved on a full statevector and on a matrix product state, and
the two are compared. With the exact reference set to the same brickwall, the only difference is
the tensor-network truncation, which at this size is none. A maximum difference at the level of
1e-8 confirms the operator factor and the sign.

In [ ]:
check = TransverseFieldIsing(; Jxx = 0.5, hz = 1.0)
cmp   = compare_caustic(check; N = 9, dt = 0.1, ttotal = 10.0)
println("max |Z_mps - Z_exact| = ", cmp.dZmax)
println("max |S_mps - S_exact| = ", cmp.dSmax)

## Boundary effect

The subtraction above removes anything tied to the boundary. To see the boundary itself, plot
the raw magnetisation `<Z_j(t)>` with open boundaries and no subtraction. A reversed light cone
propagates inward from each edge, the effect of Fig. 5.

In [ ]:
edge = evolve_caustic(spec; N = N, dt = dt, ttotal = T, init = :flip)
colormap(edge; data = edge.Z, clabel = "Z")

## Next step: the scaling exponent

The caustic above is the input to the power-law fit that extracts the exponent gamma, the
paper's headline result, gamma near 0.68 against the theoretical 2/3. The fit is left to the
reader: the dZ matrix and the time axis written by write_caustic are its only inputs, so it
is a few lines on the arrays above. NOTES.md records why a fully universal finite-size
scaling was not reached.